# Chapter 16 — LoRA and QLoRA

Chapter 4 fine-tuned a pretrained backbone by unfreezing its top layers, and
that was the right technique for a 2.2M-parameter MobileNet. Try it on a model
with seven billion parameters and you hit a wall that has nothing to do with
whether the method works:

$$ \underbrace{7\text{B} \times 4}_{\text{weights}} + \underbrace{7\text{B} \times 4}_{\text{gradients}} + \underbrace{7\text{B} \times 8}_{\text{Adam } m,\, v} \;\approx\; 104\ \text{GiB} $$

before a single activation is stored. Adam keeps two running statistics per
parameter, so **the optimizer costs twice what the model does**, and the total is
four times the weights. No consumer GPU holds that.

This chapter is the standard answer, in two halves that compose:

- **LoRA** (Hu et al., 2021): don't update the weights at all. Learn a small
  low-rank *correction* alongside them, and leave the originals frozen. The
  gradient and optimizer state then scale with the correction, not the model.
- **Quantization + QLoRA** (Dettmers et al., 2023): store the frozen weights in
  4 bits instead of 32. They are never updated, so their precision only has to
  be good enough to compute a forward pass through.

Together they take fine-tuning from "a cluster" to "one GPU", and both are small
enough to write from scratch, which is what we do here.

| Module | What you build | Dataset (Hugging Face) |
|---|---|---|
| 1 | A pretrained base model, and the memory arithmetic that motivates everything | `zalando-datasets/fashion_mnist` |
| 2 | **LoRA** from scratch: $\Delta W = BA$, zero-init, the $\alpha/r$ scale | `ylecun/mnist` |
| 3 | The rank sweep and the **which-matrices** sweep | — |
| 4 | **Merging**: why LoRA costs nothing at inference | — |
| 5 | **Quantization** from scratch: blockwise absmax, int8, and **NF4** | — |
| 6 | **QLoRA**: a 4-bit frozen base with a full-precision adapter | — |

**How each concept is presented**, the same three passes as earlier chapters:

> 🧠 **The intuition:** the idea in plain language, no symbols.
> 📐 **The math:** the same idea written precisely, so you can read papers.
> 💻 **The code:** the same idea again, executable, in the cell that follows.

**Runtime:** ~4 minutes on a GPU, for one pretraining run (~30 s) and then
sixteen short adaptation runs. Knobs are marked `# <- knob`. Both datasets are
cached from earlier chapters.


In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (7, 4.5)
print("torch", torch.__version__, "| device:", device)

---
# Module 1 — The Base Model, and the Arithmetic

## 1.1 Why fine-tuning is a memory problem before it is anything else

📐 **Count the copies.** Training any parameter with Adam requires four
tensors of its size: the weight, its gradient, and the optimizer's first and
second moment estimates. In fp32 that is **16 bytes per parameter**, and it is
the optimizer rather than the model that dominates:

| What | Bytes/param | Share |
|---|---|---|
| weights | 4 | 25% |
| gradients | 4 | 25% |
| Adam $m$ | 4 | 25% |
| Adam $v$ | 4 | 25% |

Now notice what happens if a parameter is **frozen**. No gradient, no moments,
so it costs 4 bytes and nothing else. So the memory question is not "how big is
the model" but "how many parameters are *trainable*", and those are very
different numbers if you arrange them to be.

That is the whole idea behind everything in this chapter.

💻 **The code.** We need a realistic setup: a model pretrained on one task, then
adapted to a different one. Fashion-MNIST → MNIST digits is ideal here: same
input shape, genuinely different task, both already cached.


In [ ]:
def arrays(name, n_train, n_test):
    d = load_dataset(name)
    def prep(split, n):
        s = split.shuffle(seed=42).select(range(n))
        X = np.stack([np.array(i) for i in s["image"]]).astype("float32") / 255.0
        return torch.from_numpy(X).unsqueeze(1), torch.from_numpy(np.array(s["label"])).long()
    return prep(d["train"], n_train), prep(d["test"], n_test)


Ftr, Fte = arrays("zalando-datasets/fashion_mnist", 40_000, 5_000)   # <- knob: pretraining task
Mtr, Mte = arrays("ylecun/mnist", 8_000, 5_000)                      # <- knob: adaptation task
print("pretrain on Fashion-MNIST:", tuple(Ftr[0].shape))
print("adapt to MNIST digits:    ", tuple(Mtr[0].shape), "(deliberately few images)")

## 1.2 The base model

💻 **The code.** Chapter 11's ViT, with the attention projections written as
four separate `nn.Linear` layers so Module 2 can wrap them individually. Nothing
else about it is new.

In [ ]:
class Attention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.dk = h, d // h
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.o = nn.Linear(d, d, bias=False)

    def forward(self, x):
        B, N, D = x.shape
        split = lambda t: t.view(B, N, self.h, self.dk).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(self.q(x)), split(self.k(x)), split(self.v(x)))
        return self.o(y.transpose(1, 2).reshape(B, N, D))


class Block(nn.Module):
    def __init__(self, d, h, ratio=4):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = Attention(d, h)
        self.fc1, self.fc2, self.act = nn.Linear(d, ratio * d), nn.Linear(ratio * d, d), nn.GELU()

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.fc2(self.act(self.fc1(self.n2(x))))


class ViT(nn.Module):
    def __init__(self, d=192, depth=6, h=4, n_cls=10, patch=4, img=28):
        super().__init__()
        self.patch = nn.Conv2d(1, d, patch, patch)
        n = (img // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * 0.02)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(depth)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_cls)

    def forward(self, x):
        x = self.patch(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(len(x), -1, -1), x], 1) + self.pos
        for b in self.blocks:
            x = b(x)
        return self.head(self.norm(x)[:, 0])


def train(model, data, test, epochs, lr, batch=256, params=None):
    X, y = data
    opt = torch.optim.AdamW(params if params is not None else model.parameters(),
                            lr=lr, weight_decay=0.01)
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X))
        for i in range(0, len(perm) - batch + 1, batch):
            j = perm[i:i + batch]
            loss = F.cross_entropy(model(X[j].to(device)), y[j].to(device))
            opt.zero_grad(); loss.backward(); opt.step()
    return evaluate(model, test)


@torch.no_grad()
def evaluate(model, test, batch=512):
    X, y = test
    model.eval()
    c = sum((model(X[i:i+batch].to(device)).argmax(1).cpu() == y[i:i+batch]).sum().item()
            for i in range(0, len(X), batch))
    return c / len(X)


t0 = time.time()
base = ViT().to(device)
acc_pre = train(base, Ftr, Fte, epochs=12, lr=1e-3)          # <- knob
N_PARAMS = sum(p.numel() for p in base.parameters())
print(f"base ViT: {N_PARAMS:,} parameters | Fashion-MNIST accuracy {acc_pre:.4f} "
      f"| pretrained in {time.time() - t0:.0f}s")
BASE_SD = {k: v.clone() for k, v in base.state_dict().items()}

print(f"\nfull fine-tuning this model with Adam, in fp32:")
for label, mult in [("weights", 1), ("gradients", 1), ("Adam m", 1), ("Adam v", 1)]:
    print(f"  {label:<12}{N_PARAMS * 4 * mult / 2**20:>8.2f} MB")
print(f"  {'TOTAL':<12}{N_PARAMS * 16 / 2**20:>8.2f} MB   "
      f"(the same arithmetic puts a 7B model at {7e9 * 16 / 2**30:.0f} GB)")

---
# Module 2 — LoRA

## 2.1 The hypothesis

🧠 **The intuition.** When you adapt a pretrained model to a related task, how
*much* do the weights really need to change? LoRA's bet is: not much, and not in
many independent directions. The update is a big matrix, but it should be a big
matrix of **low rank**, a handful of directions, each applied broadly.

If that is true, you never need to materialize the update as a full matrix. Two
thin matrices multiply out to it, and you train those instead.

📐 **The math.** For a frozen weight $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$:

$$ W = W_0 + \Delta W, \qquad \Delta W = \frac{\alpha}{r} B A, \qquad B \in \mathbb{R}^{d_{\text{out}} \times r},\; A \in \mathbb{R}^{r \times d_{\text{in}}} $$

with $r \ll \min(d_{\text{in}}, d_{\text{out}})$. Parameter count drops from
$d_{\text{out}} d_{\text{in}}$ to $r(d_{\text{out}} + d_{\text{in}})$, so for
$d = 192$ and $r = 8$ that is 36,864 → 3,072, a 12× reduction, and the ratio
improves as $d$ grows.

Three details that are easy to skip and all matter:

1. **$B$ is initialized to zero, $A$ randomly.** So $\Delta W = 0$ at step 0 and
   the adapted model *starts exactly as the base model*. You have met this
   exact trick twice: ResNet's zero-initialized final BatchNorm in Chapter 3,
   and adaLN-Zero in Chapter 12. Start at the identity, grow from there.
2. **Both cannot be zero**, which is a saddle point with zero gradient
   everywhere, and nothing would ever move.
3. **The $\alpha/r$ scale** decouples the learning rate from the rank. Without
   it, doubling $r$ doubles the magnitude of $BA$ and silently changes the
   effective step size, which is how rank sweeps get misread.

💻 **The code.** Twelve lines, and note that `forward` computes $B(Ax)$, never
$BA$: the low-rank product is applied as two thin matrix-vector products, so
the full $d \times d$ update is never materialized.


In [ ]:
class LoRALinear(nn.Module):
    '''Wraps a frozen nn.Linear with a trainable low-rank correction.'''

    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                       # the original weights never move
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # zero -> starts as the base

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale

    def merged_weight(self):
        return self.base.weight + (self.B @ self.A) * self.scale


ATTN_PARTS = ("q", "k", "v", "o")


def fresh_base(n_cls=10):
    '''A copy of the pretrained model with a new, randomly-initialized head.'''
    m = ViT(n_cls=n_cls).to(device)
    m.load_state_dict({k: v for k, v in BASE_SD.items() if not k.startswith("head")}, strict=False)
    return m


def apply_lora(model, targets=("q", "v"), r=8, alpha=16):
    for blk in model.blocks:
        for name in targets:
            parent = blk.attn if name in ATTN_PARTS else blk
            setattr(parent, name, LoRALinear(getattr(parent, name), r, alpha).to(device))
    return model


def adapt_lora(targets=("q", "v"), r=8, epochs=10, lr=1e-3):        # <- knobs
    torch.manual_seed(0)
    m = fresh_base()
    for p in m.parameters():
        p.requires_grad = False
    apply_lora(m, targets, r)
    for p in m.head.parameters():
        p.requires_grad = True                            # a new task needs a new head
    trainable = [p for p in m.parameters() if p.requires_grad]
    acc = train(m, Mtr, Mte, epochs, lr, params=trainable)
    return acc, sum(p.numel() for p in trainable), m


demo = apply_lora(fresh_base(), ("q", "v"), r=8)
one = demo.blocks[0].attn.q
print(f"one wrapped projection: base {tuple(one.base.weight.shape)} frozen, "
      f"A {tuple(one.A.shape)} + B {tuple(one.B.shape)} trainable")
print(f"  full matrix : {one.base.weight.numel():,} parameters")
print(f"  LoRA r=8    : {one.A.numel() + one.B.numel():,} parameters")
print(f"  delta W at initialization: max |BA| = {(one.B @ one.A).abs().max().item():.1f}  <- exactly zero")

## 2.2 The three baselines

💻 **The code.** LoRA has to be judged against both ends of the range: full
fine-tuning (the accuracy ceiling, and the memory disaster) and training only
the classifier head (the cheapest thing that could possibly work). Plus the
zero-shot number, to confirm the tasks really are different.

In [ ]:
torch.manual_seed(0)
print(f"zero-shot (no adaptation at all)   {evaluate(fresh_base(), Mte):.4f}")

m_head = fresh_base()
for p in m_head.parameters():
    p.requires_grad = False
for p in m_head.head.parameters():
    p.requires_grad = True
n_head = sum(p.numel() for p in m_head.head.parameters())
acc_head = train(m_head, Mtr, Mte, epochs=10, lr=1e-3,
                 params=[p for p in m_head.parameters() if p.requires_grad])
print(f"head only          {acc_head:.4f}   trainable {n_head:>10,}")

m_full = fresh_base()
t0 = time.time()
acc_full = train(m_full, Mtr, Mte, epochs=10, lr=3e-4)
print(f"full fine-tune     {acc_full:.4f}   trainable {N_PARAMS:>10,}   ({time.time()-t0:.0f}s)")

---
# Module 3 — Two Sweeps

## 3.1 How much rank do you actually need?

🧠 **The question.** $r$ is the one hyperparameter LoRA introduces, and the
paper's most surprising claim is that it can be astonishingly small, and they
report $r = 1$ or $2$ being competitive on GPT-3. If the low-rank hypothesis is
right, accuracy should rise steeply and then **flatten**, because the extra
directions are describing an update that does not need them.

💻 **The code.**

In [ ]:
RANKS = (1, 2, 4, 8, 16, 32)                     # <- knob
rank_acc, rank_n = [], []
print(f"{'rank':>5}{'accuracy':>11}{'trainable':>12}{'% of full':>11}")
for r in RANKS:
    a, n, _ = adapt_lora(("q", "v"), r)
    rank_acc.append(a); rank_n.append(n)
    print(f"{r:>5}{a:>11.4f}{n:>12,}{100*n/N_PARAMS:>10.2f}%", flush=True)

## 3.2 Which matrices should get an adapter?

🧠 **The question.** LoRA does not have to be applied everywhere. The original
paper attached it to $W_Q$ and $W_V$ only, and reported that spreading the same
parameter budget across more matrices beat concentrating it in fewer. That is a
testable claim and cheap to test.

Recall from Chapter 10, Module 3.1 that **two thirds of a Transformer's
parameters live in the MLPs**, not the attention, so "adapt the whole block" is
a very different proposition from "adapt attention".

💻 **The code.**


In [ ]:
TARGET_SETS = [("q",), ("q", "v"), ("q", "k", "v", "o"), ("q", "k", "v", "o", "fc1", "fc2")]
print(f"{'matrices adapted':<40}{'accuracy':>10}{'trainable':>12}")
target_acc, target_n = [], []
for tgt in TARGET_SETS:
    a, n, _ = adapt_lora(tgt, r=8)
    target_acc.append(a); target_n.append(n)
    print(f"{', '.join(tgt):<40}{a:>10.4f}{n:>12,}", flush=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(RANKS, rank_acc, marker="o", label="LoRA on $W_Q, W_V$")
axes[0].axhline(acc_full, color="tab:green", linestyle="--", label="full fine-tune")
axes[0].axhline(acc_head, color="tab:red", linestyle=":", label="head only")
axes[0].set_xscale("log", base=2); axes[0].set_xlabel("rank r"); axes[0].set_ylabel("accuracy")
axes[0].set_title("Accuracy saturates in rank"); axes[0].legend(fontsize=8); axes[0].grid(True)

axes[1].plot(target_n, target_acc, marker="s", color="tab:purple")
for n, a, t in zip(target_n, target_acc, TARGET_SETS):
    axes[1].annotate(",".join(t), (n, a), fontsize=7, xytext=(3, -9), textcoords="offset points")
axes[1].axhline(acc_full, color="tab:green", linestyle="--", label="full fine-tune")
axes[1].set_xscale("log"); axes[1].set_xlabel("trainable parameters"); axes[1].set_ylabel("accuracy")
axes[1].set_title("Where you spend the budget matters"); axes[1].legend(fontsize=8); axes[1].grid(True)
plt.tight_layout(); plt.show()

### Reading the sweeps

**Rank saturates, and it saturates early.** Most of the achievable gain is
present by $r = 4$–$8$, and quadrupling the rank after that buys very little.
That is the low-rank hypothesis surviving a test: if the required update genuinely
had high rank, accuracy would keep climbing with $r$, and it does not.

**Spreading beats concentrating.** Adapting all four attention projections at
$r = 8$ beats adapting $W_Q$ alone at the same rank by a wide margin, and it
reproduces the paper's finding. If you have a fixed budget, prefer more matrices
at lower rank over fewer matrices at higher rank.

**And LoRA does not quite reach full fine-tuning here.** It should not be
expected to: this is a *hard* adaptation (Fashion-MNIST to digits is close to a
different domain, not a nearby one) with few images, and the low-rank constraint
is a real constraint. The honest framing is a trade, not a free lunch: a small
fraction of the accuracy gap for one to two orders of magnitude fewer trainable
parameters.


---
# Module 4 — Merging: Why LoRA Is Free at Inference

🧠 **The intuition.** A LoRA layer computes $W_0 x + BAx$, which is two extra
matrix multiplies per layer, so real latency, and it would be an odd thing to
pay forever. But addition is addition: since both terms are linear in $x$,

$$ W_0 x + \tfrac{\alpha}{r} BAx = \Big(W_0 + \tfrac{\alpha}{r} BA\Big) x $$

you can add the update **into the weights once**, after training, and serve an
ordinary model. Zero extra parameters, zero extra latency, no LoRA code in the
serving path.

This is why adapters are the deployment format of choice: a 30 MB adapter file
per customer, merged into one shared base model at load time. Adapter methods
that insert *nonlinear* modules cannot do this, and that is most of why LoRA won.

💻 **The code.** Do not take "mathematically identical" on faith.


In [ ]:
_, _, lora_model = adapt_lora(("q", "k", "v", "o"), r=8)
acc_unmerged = evaluate(lora_model, Mte)

merged = fresh_base()
merged.load_state_dict({k: v for k, v in lora_model.state_dict().items()
                        if "A" not in k and "B" not in k and "base." not in k}, strict=False)
with torch.no_grad():
    for mb, lb in zip(merged.blocks, lora_model.blocks):
        for name in ATTN_PARTS:
            getattr(mb.attn, name).weight.copy_(getattr(lb.attn, name).merged_weight())

acc_merged = evaluate(merged, Mte)
x = Mte[0][:64].to(device)
with torch.no_grad():
    gap = (lora_model(x) - merged(x)).abs().max().item()

print(f"LoRA model (adapters live) : {acc_unmerged:.4f}")
print(f"merged into the weights    : {acc_merged:.4f}")
print(f"max |logit difference|     : {gap:.3e}")
print(f"\nmerged model has {sum(p.numel() for p in merged.parameters()):,} parameters "
      f", identical to the base, with no LoRA modules anywhere")

---
# Module 5 — Quantization from Scratch

## 5.1 The idea, and the two decisions

🧠 **The intuition.** LoRA fixed the *gradient* and *optimizer* memory. The
frozen weights are still sitting there in fp32, and for a 7B model that is 28 GB
doing nothing but being read. But they are never updated, so they only need
enough precision to compute a forward pass, not to accumulate small updates.
Four bits turns out to be enough.

Quantization means picking a small set of representable values and snapping every
weight to the nearest one. Two decisions follow:

**How do you scale?** Weights vary in magnitude across a tensor, so one global
scale wastes most of the range on outliers. **Blockwise** quantization splits the
tensor into small blocks (64 values) and gives each its own scale, so one
extreme weight only ruins its own block. The scales are stored in fp32 and
themselves cost memory, which we count.

**Where do you put the levels?** Uniformly spaced levels are the obvious choice
and the wrong one. Neural network weights are approximately **normally
distributed**, so uniform levels put as many codes far out in the tails, where
almost no weights live, as near zero, where almost all of them do.

📐 **The math.** For block $b$ with scale $s_b = \max_i |w_i|$:

$$ q_i = \arg\min_k \Big| \frac{w_i}{s_b} - \ell_k \Big|, \qquad \hat{w}_i = s_b \cdot \ell_{q_i} $$

**NF4** chooses the levels $\ell_k$ to be the **quantiles of a standard normal**,
so that a normally-distributed input uses all 16 codes about equally often,
which is information-theoretically optimal for that distribution. That is the
whole content of "4-bit NormalFloat".

💻 **The code.** Both schemes, and a uniform 4-bit control so the NF4 claim is
actually tested rather than assumed.


In [ ]:
# The 16 NF4 levels: quantiles of a standard normal, rescaled to [-1, 1].
NF4 = torch.tensor([-1.0, -0.6961928, -0.5250731, -0.3949175, -0.2844414, -0.1847734,
                    -0.0910500, 0.0, 0.0795803, 0.1609302, 0.2461123, 0.3379152,
                    0.4407098, 0.5626170, 0.7229568, 1.0], device=device)
INT8 = torch.arange(-127, 128, device=device).float() / 127.0        # uniform, 8-bit
UNIFORM4 = torch.linspace(-1, 1, 16, device=device)                  # uniform, 4-bit (the control)


def quantize(w, levels, block=64):                                   # <- knob: block size
    flat = w.flatten()
    pad = (-flat.numel()) % block
    blocks = F.pad(flat, (0, pad)).view(-1, block)
    absmax = blocks.abs().amax(1, keepdim=True).clamp(min=1e-8)      # one scale per block
    idx = (blocks.unsqueeze(-1) / absmax.unsqueeze(-1) - levels).abs().argmin(-1)
    return idx.to(torch.uint8), absmax, w.shape, pad


def dequantize(idx, absmax, shape, pad, levels):
    out = (levels[idx.long()] * absmax).flatten()
    return out[: out.numel() - pad].view(shape)


W = torch.cat([p.detach().flatten() for p in base.parameters() if p.dim() == 2])
print(f"measured on {W.numel():,} real weights of the pretrained ViT\n")
print(f"{'scheme':<26}{'bits':>6}{'rel. error':>13}")
for name, levels, bits in [("int8, uniform", INT8, 8),
                           ("4-bit, uniform (control)", UNIFORM4, 4),
                           ("NF4, normal quantiles", NF4, 4)]:
    q = quantize(W, levels)
    err = (dequantize(*q, levels) - W).abs().mean().item() / W.abs().mean().item()
    print(f"{name:<26}{bits:>6}{err:>12.2%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sample = W[torch.randperm(W.numel(), device=W.device)[:200_000]].cpu().numpy()
axes[0].hist(sample, bins=200, density=True, alpha=0.6, label="ViT weights")
axes[0].set_title("Weights are approximately normal"); axes[0].set_xlabel("value")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for y, (lv, name) in enumerate([(UNIFORM4, "uniform 4-bit"), (NF4, "NF4")]):
    axes[1].scatter(lv.cpu().numpy(), [y] * len(lv), s=40, label=name)
axes[1].set_yticks([0, 1]); axes[1].set_yticklabels(["uniform", "NF4"])
axes[1].set_xlabel("level position"); axes[1].set_title("Where the 16 codes go")
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("NF4 clusters its codes near zero, where the weights actually are.")
print(f"levels within |x| < 0.3, uniform: {(UNIFORM4.abs() < 0.3).sum().item()}/16"
      f"   NF4: {(NF4.abs() < 0.3).sum().item()}/16")

**NF4 beats uniform 4-bit, and by less than the marketing implies.** The
improvement is real and reproducible, and it comes entirely from putting the
codes where the mass is, but at this model scale it is a modest relative gain
rather than a transformation. Both 4-bit schemes are far behind int8, which is
the honest picture: going from 8 bits to 4 costs real accuracy, and NF4 recovers
part of that cost rather than eliminating it.

The reason to accept the cost anyway is in the next module.


---
# Module 6 — QLoRA

🧠 **The intuition.** The two techniques are orthogonal, and they compose
exactly:

- The **frozen base** is only ever read in the forward pass → store it in 4 bits.
- The **LoRA adapters** receive gradients and optimizer state → keep them in
  full precision, because they are tiny.

That is QLoRA. The 4-bit weights are dequantized on the fly inside the matmul:
in a real implementation, inside the CUDA kernel, so the fp16 copy never exists
in memory. Ours dequantizes in `forward`, which measures the accuracy cost
faithfully while simulating the storage saving.

📐 **What we are trading.** Quantization error is a fixed perturbation to a
frozen function. The adapter is trained *through* that perturbed function, so it
can partly compensate for it, which is the subtle reason QLoRA works better than
"quantize a fine-tuned model" would suggest.

💻 **The code.**


In [ ]:
class QLoRALinear(nn.Module):
    '''A 4-bit frozen weight matrix with a full-precision low-rank adapter.'''

    def __init__(self, lin: nn.Linear, r=8, alpha=16, levels=NF4, block=64):
        super().__init__()
        idx, absmax, shape, pad = quantize(lin.weight.detach(), levels, block)
        self.register_buffer("idx", idx)              # uint8 codes
        self.register_buffer("absmax", absmax)        # fp32 scale per block
        self.shape, self.pad, self.levels = shape, pad, levels
        self.bias = lin.bias
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(r, lin.in_features, device=device)
                              / math.sqrt(lin.in_features))
        self.B = nn.Parameter(torch.zeros(lin.out_features, r, device=device))

    def forward(self, x):
        W = dequantize(self.idx, self.absmax, self.shape, self.pad, self.levels)
        return F.linear(x, W, self.bias) + F.linear(F.linear(x, self.A), self.B) * self.scale


def adapt_qlora(targets=("q", "k", "v", "o"), r=8, epochs=10, lr=1e-3, levels=NF4):
    torch.manual_seed(0)
    m = fresh_base()
    for p in m.parameters():
        p.requires_grad = False
    for blk in m.blocks:
        for name in targets:
            setattr(blk.attn, name, QLoRALinear(getattr(blk.attn, name), r, 16, levels).to(device))
    for p in m.head.parameters():
        p.requires_grad = True
    trainable = [p for p in m.parameters() if p.requires_grad]
    return train(m, Mtr, Mte, epochs, lr, params=trainable), sum(p.numel() for p in trainable)


acc_lora_ref, n_ref, _ = adapt_lora(("q", "k", "v", "o"), r=8)
acc_q8, _ = adapt_qlora(levels=INT8)
acc_q4, n_q = adapt_qlora(levels=NF4)

print(f"{'method':<34}{'accuracy':>10}{'trainable':>12}")
print(f"{'full fine-tune (fp32)':<34}{acc_full:>10.4f}{N_PARAMS:>12,}")
print(f"{'LoRA r=8, fp32 base':<34}{acc_lora_ref:>10.4f}{n_ref:>12,}")
print(f"{'QLoRA r=8, int8 base':<34}{acc_q8:>10.4f}{n_q:>12,}")
print(f"{'QLoRA r=8, NF4 base':<34}{acc_q4:>10.4f}{n_q:>12,}")

In [ ]:
print(f"storage for the frozen base ({N_PARAMS:,} parameters):\n")
print(f"{'format':<10}{'MB':>9}{'vs fp32':>10}")
for name, bits, scale_overhead in [("fp32", 32, 0), ("fp16", 16, 0),
                                   ("int8", 8, 32 / 64), ("NF4", 4, 32 / 64)]:
    mb = N_PARAMS * (bits + scale_overhead) / 8 / 2**20
    print(f"{name:<10}{mb:>9.2f}{N_PARAMS * 4 / 2**20 / mb:>9.1f}x")

full_mb = N_PARAMS * 16 / 2**20
qlora_mb = N_PARAMS * (4 + 32/64) / 8 / 2**20 + n_q * 16 / 2**20
print(f"\ntraining footprint (weights + grads + Adam states):")
print(f"  full fine-tune : {full_mb:>8.2f} MB")
print(f"  QLoRA          : {qlora_mb:>8.2f} MB   ({full_mb / qlora_mb:.1f}x smaller)")
print(f"\nthe same arithmetic on a 7B model: "
      f"{7e9 * 16 / 2**30:.0f} GB  ->  {(7e9 * 4.5 / 8 + n_q / N_PARAMS * 7e9 * 16) / 2**30:.1f} GB")

### Reading the result

**QLoRA costs a little accuracy and saves a great deal of memory**, and both
halves of that sentence are visible above. The NF4 base lands slightly below
the fp32-base LoRA, a real cost, honestly reported, in exchange for storing
the frozen model at roughly **one seventh** the size.

Two caveats worth carrying, because our setup flatters the method in one way and
penalizes it in another:

- **Penalizing:** our model is 2.7M parameters. Quantization error is relatively
  *worse* on small models, since there are fewer weights per block for the scale
  to amortize over, and less redundancy to absorb the noise. The QLoRA paper's
  result (4-bit matching 16-bit fine-tuning on 65B models) is at a scale where
  that redundancy is enormous.
- **Flattering:** we dequantize in `forward` and then run a normal fp32 matmul,
  so we measure the *accuracy* cost honestly but not the *speed* cost. Real
  4-bit inference needs a kernel that dequantizes inside the matmul; done
  naively it is slower than fp16, not faster. Memory is the win here, not
  throughput.

The 7B extrapolation in the last line is the entire point of the chapter: the
same model that needed over 100 GB to fine-tune fits, with these two techniques, on a
single consumer card.


---
# Wrap-Up

| You built | The transferable lesson |
|---|---|
| The Adam memory arithmetic | Training costs 16 bytes/parameter, and 12 of them are optimizer. Frozen parameters cost 4 |
| `LoRALinear` | $\Delta W = \frac{\alpha}{r} BA$ with $B$ zero-initialized, so the adapted model starts *as* the base, the third appearance of "begin at the identity" after Chapter 3 and Chapter 12 |
| The $\alpha/r$ scale | Decouples learning rate from rank; without it a rank sweep silently sweeps the step size too |
| The rank sweep | Accuracy saturates by $r \approx 4$–$8$, which is the low-rank hypothesis tested rather than assumed |
| The target sweep | Spread a fixed budget across more matrices at lower rank, not fewer at higher rank |
| Merging | $W_0 + \frac{\alpha}{r}BA$ is a weight matrix. Zero inference cost, verified to floating-point precision, and the reason adapters are the deployment format |
| Blockwise absmax + NF4 | One scale per 64 weights; put the codes at normal quantiles because weights are normal. Beats uniform 4-bit, by less than advertised |
| QLoRA | 4-bit frozen base + full-precision adapter. A few points of accuracy in exchange for storing the base at about one seventh the size |

**The pattern underneath all of it.** Every technique here comes from asking
*which tensors actually need to be precise, and which need to be trainable*, and
then refusing to pay for the rest. The weights need precision but not gradients.
The adapters need gradients but are tiny. The optimizer state is the largest
thing in a naive setup and is pure overhead on frozen parameters.

**One thing this chapter did not fix.** All the savings above are on
*parameters*. The other half of a training run's memory is **activations**, and
for a Transformer the single largest activation is the $n \times n$ attention
score matrix, which no amount of adapter cleverness touches. Chapter 15 hit the
same wall from the architecture side and measured attention's memory growing 4×
per doubling of sequence length.

That tensor is the last big target, and it turns out you can avoid
materializing it at all.
